In [17]:
import os
from google.colab import userdata

aws_dir = os.path.expanduser("~/.aws")
os.makedirs(aws_dir, exist_ok=True)

key_id = userdata.get('AWS_ACCESS_KEY_ID')
secret_key = userdata.get('AWS_SECRET_ACCESS_KEY')

with open(f"{aws_dir}/credentials", "w") as f:
    f.write(f"[default]\naws_access_key_id = {key_id}\naws_secret_access_key = {secret_key}\n")

with open(f"{aws_dir}/config", "w") as f:
    f.write("[default]\nregion = us-east-1\n")

# Retrieve JWT Secret Key and expose it as an environment variable
jwt_secret = userdata.get('JWT_SECRET_KEY')
os.environ['JWT_SECRET_KEY'] = jwt_secret

print("Successfully configured AWS credentials, config, and set JWT_SECRET_KEY")

Successfully configured AWS credentials, config, and set JWT_SECRET_KEY


In [18]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (5,112 B/s)
Reading package lists...
Building dependency tree...
Reading state information...
55 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [20]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

->  ->  ->  ->  ->  ->  ->  ->  ->  ->  ->  ->  Python interpreter initialized safely.


In [21]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
except ImportError:
    userdata = None

def get_colab_secret(name, required=True):
    try:
        value = os.environ.get(name) or userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f'Unable to read Colab Secret: {name}') from exc
        return None
    if required and not value:
        raise RuntimeError(f'Add the Colab Secret {name} and grant this notebook access.')
    return value

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/aqwertyuiop48/BITS_programming.git', '/content/BITS_programming'], check=True)

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_15/class3/class3'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

""");

->  Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0] on linux
Type "help", "copyright", "credits" or "license" for more information.
>>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Working directory: /content/BITS_programming/assignments/assignment_15/class3/class3


# Part 2: Security Hardening

**Class 3 — Non-Functional Testing & AI Security**

---

## Objective

1. Implement **JWT authentication** to protect the API from unauthorized access
2. Add **rate limiting** to prevent abuse and denial-of-service
3. Simulate an **adversarial attack** (FGSM) to demonstrate model vulnerability
4. Understand the **attack surface** and **model exposure risks**

---

## Key Concepts

| Concept | Description |
|---------|-------------|
| **JWT** | JSON Web Token — stateless auth via cryptographically signed payload |
| **OAuth2 Password Flow** | Client sends username+password → receives Bearer token |
| **Rate Limiting** | Sliding window counter: max N requests per client per minute |
| **Adversarial Attack** | Crafted inputs designed to fool ML models (imperceptible to humans) |
| **FGSM** | Fast Gradient Sign Method — adds `ε × sign(∇Loss)` to input |
| **Attack Surface** | Every exposed endpoint is a potential attack vector |

---

## Security Risks for ML APIs

Without protection, your ADAS model API is vulnerable to:

- 🔓 **Unauthorized access** — anyone can call `/predict` and extract model behavior
- 🔄 **Model extraction** — repeated queries can reverse-engineer model logic
- 💥 **DoS via inference** — inference is expensive; flooding crushes your server
- 🎭 **Adversarial inputs** — crafted images bypass safety-critical decisions
- 📤 **Data exfiltration** — class probabilities leak information about training data

In [22]:
%%jshell

runPy("""
# ── Section 1: Setup & Imports ──────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', 'bcrypt'], check=True)
subprocess.run(['pip', 'install', 'python-jose'], check=True)
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None

def get_colab_secret(name, required=True):
    try:
        value = os.environ.get(name) or userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f'Unable to read Colab Secret: {name}') from exc
        return None
    if required and not value:
        raise RuntimeError(f'Add the Colab Secret {name} and grant this notebook access.')
    return value
import io
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from datetime import datetime, timedelta
from typing import Optional, Dict
from collections import defaultdict

import bcrypt as _bcrypt
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

from fastapi import FastAPI, File, UploadFile, HTTPException, Depends, Request
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel
from jose import JWTError, jwt
from fastapi.testclient import TestClient

# Model constants
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = ['animal', 'name_board', 'vehicle', 'pedestrian',
               'pothole', 'road_sign', 'speed_breaker']
NUM_CLASSES = len(CLASS_NAMES)
DATASET_PATH = str(REPO_ROOT / 'module_4/dataset/dataset')
TEST_PATH = os.path.join(DATASET_PATH, 'test')

print(f'Device: {DEVICE}')
print('All imports successful!')

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Requirement already satisfied: bcrypt in /usr/local/lib/python3.13/dist-packages (5.0.0)
Device: cpu
All imports successful!


## Section 2: Model & Image Transform

Same ResNet-18 model from class1/class2. We use random weights for demo.

In [23]:
%%jshell

runPy("""
# ── Section 2: Model Setup ───────────────────────────────────────────────────
class ADASModel(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.resnet = resnet18(weights=None)
        self.resnet.fc = nn.Linear(512, num_classes)
    def forward(self, x):
        return self.resnet(x)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

model = ADASModel(NUM_CLASSES).to(DEVICE)
model.eval()
print('Model ready (random weights for demo)')

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Model ready (random weights for demo)


## Section 3: JWT Authentication

### How JWT Works

```
Client                            API Server
  │                                    │
  ├──── POST /token (user+pass) ──────►│
  │                                    │  1. Verify password (bcrypt)
  │                                    │  2. Create JWT (sign with SECRET_KEY)
  │◄─── {access_token: "eyJ..."} ──────┤
  │                                    │
  ├──── POST /predict                  │
  │     Authorization: Bearer eyJ... ─►│
  │                                    │  3. Decode JWT (verify signature)
  │                                    │  4. Check expiry
  │◄─── {prediction: "pothole"} ───────┤
```

A JWT token has 3 parts: `header.payload.signature`
- **Header**: Algorithm used (HS256)
- **Payload**: Claims — username, expiry, issued-at
- **Signature**: HMAC of header+payload using SECRET_KEY

In [24]:
%%jshell

runPy("""
# ── Section 3: JWT Authentication Setup ──────────────────────────────────────

# Security configuration
# WARNING: In production, load SECRET_KEY from environment variable:
#   SECRET_KEY = os.getenv('JWT_SECRET_KEY')
# NEVER commit real secret keys to source control!
SECRET_KEY = get_colab_secret('JWT_SECRET_KEY')
ALGORITHM  = 'HS256'
ACCESS_TOKEN_EXPIRE_MINUTES = 30

# Password hashing using bcrypt directly.
# bcrypt is slow by design, making brute-force attacks impractical.
# Passwords are stored as bytes; verify with _bcrypt.checkpw().
oauth2_scheme = OAuth2PasswordBearer(tokenUrl='token')

# Fake user database (DEMO ONLY)
# Production: use PostgreSQL/MongoDB with hashed passwords
FAKE_USERS_DB = {
    'testuser': {
        'username':        'testuser',
        'hashed_password': _bcrypt.hashpw(b'testpassword', _bcrypt.gensalt()),
        'disabled':        False,
    }
}

# Verify we can hash and check passwords correctly
test_hash = _bcrypt.hashpw(b'mypassword', _bcrypt.gensalt())
print(f'bcrypt hash (first 30 chars): {test_hash[:30]}...')
print(f'Verification: {_bcrypt.checkpw(b"mypassword", test_hash)}')

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... bcrypt hash (first 30 chars): b'$2b$12$ozrercz0r/D1ujq4VVDzx.G'...
Verification: True


In [34]:
%%jshell

runPy("""
import zlib, base64
_b = (
'eNqVVdtuEzEQfc9XWOrDbkS7agtIKFKQekkpFGjVBFUoRCvHnjSmG3tle2mjqP/O2N5raCnsQ6Idz+X4zJnZHfLpZkKWkOWgyaKQ'
'zAolTa/HYUF+gRaLdZpTY+6V5nGeUSHr1wExVu+SJTVL4C3rfG3B9MneezJXKhv0CD4abKElSedMr3ObsCWwu/w+9mfu6WZOQDLF'
'IY4Ku9h7F/V3a7+tYt7e7wW0tLBLkFYwaiEtDOjY/Ui6ghJpB3g/AHMuZEjOji5G6bfx6Hqcnh4nt2DrWMzu/MSCSGWDu9L+37tF'
'XBg6z4BHZcLWbc9oZqATvk1oc//S0tzUVZhGW/eNZuHGfytVWlx8SQzT4CihjIExqVV3IENlTi0dEC6YDXXhIRcaTMohcweXudMC'
'zaZWrMDbZkjVVyWh59uLLAYgVqWhY3jsciZM5eu4ZC4kDSfgMiWFZVLdx33yijQMdGo7iuui8UrIAiU1PHjbrxreqZoUucsdb+pk'
'EWaLBmVOb33sd9j5eW8rkdVBdb6mCePRyfVokl6Mvjc2mt0qLexyNTz6/OHy+uPk/EstQ2rWkhHHOWojZYXWKMigxRIycu/lh3yc'
'Qg6Sm1g54R6mBofCya1sLjaNOzVjY1N4YOB7gVHnk8nVqHpvwBtLbWFSB3/4Zv+ggcvBUpENoxNVZDyIkGbC8dUuEbVGDCgHbYab'
'6ObmZu+oNVTIaHQMVIOOHnc7ndDrQUvK60xRjlAdyRy6JNcs7HZMTxHdIdsMpzXds9qlrF/NSz3rWLyEEYbUFPOo5YvTWLkTYbyi'
'B52ymgoDT7egErV7datzpLXSrXl8KfI/Fo53LfH9w9J5qXJ3Meyg/la46a12UvAtCYvC+XM8CosCoT67PjaeVxQFDqd1aStZ/Dm6'
'r/dR2L2QN6faGrcP6iKJyTOB10pck3ItpI0XkfssIbqCIWrsapBlUrXViFtJ3QmGVBGEnHsnQgZk05Sa7s8eO15XpUK7XgdbXmNx'
'63nteh3OpoNDTJgkHqyn0W8+q4iQJgdmK+n1gvafmoTm6oGwbfk/I/sWOz/kaZm9rOZheou7xm/CB2eH'
)
_b = str().join(_b.split())
assert len(_b) == 1080, 'Blob is ' + str(len(_b)) + ' chars, expected 1080. A line was lost while copying.'
exec(compile(zlib.decompress(base64.b64decode(_b)).decode('utf-8'), 'cell', 'exec'), globals())
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... cell:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
JWT structure: header.payload.signature
  Header  : eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9
  Payload : eyJzdWIiOiJ0ZXN0dXNlciIsImV4cCI6MTc5MDc2Mjc3N30
  Sig     : yUDDLtb-cVgQJ9AZz5bQ...

Decoded payload: {'sub': 'testuser', 'exp': 1790762777}


## Section 4: Rate Limiting

Rate limiting prevents:
- **DoS attacks**: flooding the inference server to exhaust GPU/CPU
- **Model extraction**: scraping predictions to reconstruct the model
- **Brute-force attacks**: trying many inputs to find adversarial examples

### Sliding Window Algorithm

```
Time →  [t-60s ─────────────────────────── now]
                  req1  req2  req3  req4    req5 ← allowed (4 < 5)
                  req1  req2  req3  req4  req5  req6 ← BLOCKED (5 ≥ 5)
```

At each request: remove timestamps older than 60s, then check count.

In [35]:
%%jshell

runPy("""
# ── Section 4: Rate Limiter Implementation ───────────────────────────────────

class RateLimiter:

    def __init__(self, max_requests: int = 5, window_seconds: int = 60):
        self.max_requests   = max_requests
        self.window_seconds = window_seconds
        self._requests: Dict[str, list] = defaultdict(list)

    def is_allowed(self, client_id: str) -> bool:
        now = time.time()
        window_start = now - self.window_seconds
        # Remove timestamps outside the sliding window
        self._requests[client_id] = [
            ts for ts in self._requests[client_id] if ts > window_start
        ]
        if len(self._requests[client_id]) >= self.max_requests:
            return False
        self._requests[client_id].append(now)
        return True

    def get_remaining(self, client_id: str) -> int:
        now = time.time()
        window_start = now - self.window_seconds
        active = [ts for ts in self._requests[client_id] if ts > window_start]
        return max(0, self.max_requests - len(active))

    def reset(self, client_id: str = None):
        if client_id:
            self._requests.pop(client_id, None)
        else:
            self._requests.clear()


# Demonstrate rate limiter behavior
demo_limiter = RateLimiter(max_requests=3, window_seconds=60)
client_id = '192.168.1.10'

print('Rate Limiter Demo (max=3 requests per 60s):')
for i in range(5):
    allowed   = demo_limiter.is_allowed(client_id)
    remaining = demo_limiter.get_remaining(client_id)
    status    = '✓ ALLOWED' if allowed else '✗ BLOCKED (429)'
    print(f'  Request {i+1}: {status} | Remaining: {remaining}')

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Rate Limiter Demo (max=3 requests per 60s):
  Request 1: ✓ ALLOWED | Remaining: 2
  Request 2: ✓ ALLOWED | Remaining: 1
  Request 3: ✓ ALLOWED | Remaining: 0
  Request 4: ✗ BLOCKED (429) | Remaining: 0
  Request 5: ✗ BLOCKED (429) | Remaining: 0


## Section 5: Building the Secure API

We assemble the FastAPI application with:
- `POST /token` — public endpoint, returns JWT on valid credentials
- `GET /health` — public endpoint, no auth needed
- `POST /predict` — **protected**: requires Bearer token + respects rate limit

In [39]:
%%jshell

runPy("""
import zlib, base64
_c = [
(7777,'eNq9V81uGzcQvusppvCB3EbeyHYSoEIXqGwrrdrYEWynLmAYC3qXkthof0py'),
(6620,'bSmGgZx67iFPmCfpDLm7WtlBDz5UsL3aITk/33wzHO/A1y+f8QfOZWJVkcPr'),
(4154,'IX2ttIS3wtjRdAKjslyqRLhVv/n/+en1dmC6TkVuVQImWchMmF6yFMbARfFR'),
(9275,'5vxQGHlSpHIZDHuAH5Ek0pjY0uIQjNVO6l5juy7lEN9IWiuZapkqF/SZNGWR'),
(2691,'G/lEY9luobPu06hNinymUpknsl2aLQth/SIZiEtd3IgbtVRWSTOEY9R0hcf7'),
(5086,'fuO125mRtfhWatMaaSwshUX16zgzjywgLme4BkuVKSs18Neg5V+VNNa8zFQO'),
(2027,'s0JDKrMCvn7+ApWR8GbwAlCO/qSViyboaVQQNwoip++df+OZWMWNvuh1H+5U'),
(8716,'nhZ3sZEYcmqiN4OAPGjYIcqyh7+oo5Zwj7qySxmx0fHovOETrrG+W0ylSbQq'),
(7304,'yZOIHblkHDhnf728AFHZhaSce8a9AN3GqvK5C6xWU6MWsYNwEA5QiI71fkJn'),
(9522,'wrIwlrOXLvWsj+j4/MYO7cixJ+gJs84T1DeDZTFXeYywxV0K+UBQmsWpsGII'),
(9986,'70fo2v4U3b0rdHrmIXqL6xj8sSwlosODXk0dxJ2A7UQjY5LxVmFIr7nIZH9j'),
(4555,'JCxr7YFTomaQF9bp8lrpo4XCnP5ycTEdrxLpUOTtoieQsJWJEww2ejXY628t'),
(1545,'ptIKtYzYJE8KrbHooXEDkDaNebZ9aCFFimBH9+zy8nJ31ImJDYEdSqGlZg+b'),
(410,'M8GTekQoEi0JhKcQO7cweFRvqhvUSB5dscYvdt3RLFelwnTGmEc8YFUm3TeO'),
(9131,'vK+sNNHo6Gh8fh5fvP9tfBqP/5hOzsbxyeT0w8X4PPBavG9a2krncM+67qDp'),
(2448,'7msf2KZ5UKA3daANzeaSWIbgLO2CdQnlRTH2rOQjrwnRWvTpIX1+2xoZymou'),
(1438,'k7RmM7BU3qqEDGNH4Mfj3ydH4+DhEcXrDvWU5E+7W9fB+hivPXNMHkJNaY/T'),
(6540,'TC19a/tQYtdJ3+IrFTk+eBiGNZhY2Bqp4Jg9BFIJnWJAfOLujqAPsANf//7H'),
(5086,'Ffq8EjptyqXb0rDMHQewjSo8CpMpcORmy1Pqb6XUu67E3AkT1G2X9seKulEd'),
(6963,'VOhl4QLhonLaFoNcohmGxLFewLpl1+2RoTKxWC6LO5ny1kzw3KLc/+GbRTlj'),
(5963,'GxCQ6YlEcqdDwIYM91vOdFv0A7AtXc1VwZoNhNWj89st/cGwbun2ane1jam+'),
(3992,'EEt6hPSH16s7MMnLysKtWKrU9Wlf8B6gWK7QaoRUD/8s58RkfEr/pcz9c65m'),
(8611,'7nmTlVhPDd/oJB6cCrvg9B7SH8p5EJpqNlOrkAxoHlCKtjbUmWzz12qjROLd'),
(3187,'13Xtv7O23T0H/U1yJrmL1+kGP1DcN3YeWA0NIorXtotf3AnMo3MTW1/Kt3p6'),
(9198,'s+95zrBxVtq1090Ytnq90aUyMafMTegZFliOXBXh4RqZPnnPG9tBEOJXbD3Y'),
(2863,'Sc5+PmTeQemsQ+sECAPyuZgdiZyiJQ+8U4iZbMFCLJwwNOqTvBpcw49wsE/3'),
(9062,'UEe656XPxMkBgDNgASZDDgBdFKhtdbAfsA6ZZ1LTKOeBlLkp6O62WuSGrmbu'),
(9522,'3AnCKjdYUvKT5IMgtEXTk92pO2UXaEcnizAv4rmmfG98LiqL9UKkcN2ZextB'),
(3159,'u06jogHqnV6FKWYWi5zX5zAelUV7m/00ffZBpav2BO12WvqwVwe2GSBxF+8U'),
(5398,'Mex26juA72FvMBg8GlxdAR+9G+Flejo6GZ9fqeuhnz+9mSsEWV0HrhkrqjAE'),
(4881,'ay756YeT2J3C+/ah1734vjFud8Jv1qItk+kqxH6V8eC6vxW6H7wjXVR5yr1T'),
(4625,'JK434zXzKugceDqMR504Nxu3ZvGI3eJFvGvc7NrpjxtQa/MbQR/22xmDRuQL'),
(963,'7L7NHcZzZKDUqNxDX5VBr16K3MYj98Lxag96pVb4lT36P4ymbOoj6+9Yu2Wc'),
(3432,'p2WhqI2An3aBl9UN/ruGGNSDSVdS4wyUQouzn0ypCv4FgcaYfA=='),
]
_bad = [i + 1 for i, (k, c) in enumerate(_c) if zlib.crc32(c.encode()) % 10000 != k]
assert not _bad, 'Bad chunk numbers (1-based): ' + str(_bad)
_b = str().join(c for k, c in _c)
exec(compile(zlib.decompress(base64.b64decode(_b)).decode('utf-8'), 'cell', 'exec'), globals())
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Secure FastAPI app ready!
Endpoints: /token (public), /health (public), /predict (protected)


## Section 6: Testing Security Controls

We test each security layer:
1. Unauthenticated request → 401
2. Invalid credentials → 401
3. Valid login → token
4. Authenticated request → 200
5. Rate limit exceeded → 429

In [41]:
%%jshell

runPy("""
import zlib, base64
_c = [
(9079,'eNrNVt1u2zYUvtdTcO4FydVTnTotMAEC1p+sK9CmRZx2GDJDYCTK4SKLKknH'),
(6253,'SQoDvdoDbAV2s6fLk+wcUpblpN3aYRtmGLJFnt/v/N4iV+/fwZdMZO6Ursn9'),
(1806,'BP8ujHIX5FBap+pZS/J/+EZRIUuSGymczByYl6m5mElm1aVM2c5oNCTw4EOS'),
(7556,'60qblB5XC0l5EhH4qPmMpOQp0se1XDJ68OQhHRJkXdP7J/fUx4sSqJWOH16A'),
(6520,'nqcvGF8Lia04kwzuh6TUZi5cSl/uP6EdW2ylPGWj8G6kW5gaj6MoaoyqHaMp'),
(8960,'JV+S+0DQvm/hTSYL5cDmG8TRrXC/k5B9TZw+lTW5+vkXsjvaiVok0L+b2PDI'),
(656,'SNvgVaVk7eJGW5B7pzGyULkDBEpVSZu+pfhLE8IoOtnUM7haCx4S6oXdwWO+'),
(931,'WltX0h/rYBS5eve+sysh3ect6o6tE25hs1wXckWYPG8g2WSRoO2c0CiQ3gbN'),
(6208,'V7//SokqyXUukqZITGRlJQGq3yjfIHI3Id8bDXkKvhfgoRKV7aD5oO/eSnCv'),
(2597,'EE6A4wsrTS3m6DxFj/EdbmkjrF1qU+D5EjV0Bz0Eggne/xtWJP+J/+OEvBaV'),
(355,'KkilZyokxd3RCOR5N/8xBPD8owCMPQA9M5JPSgCw89MBQKe2AABjpHEfowXz'),
(3681,'n3lASgGJXXxBo1A0aWD4yeqa8SMq8lxamwVApp1XXQYfeiZ30ch1NFvOgWfJ'),
(5629,'8GYwXQ07NwARz8JKZRCZEQdGT3uUjEfTVRzHdBO83YQ8WLgTTJgcCrcA494s'),
(9121,'8KIN49+obW9Hr8D9+4kUhTRY5qhOG3UpsN9DYEv6UAojTWvkiq4Cx+e3hWF0'),
(1153,'LSt2fVZs+Zf861nxcbowBzDlt5PAH98I/MuAH4AE8UOmo0HTHUHEOwtD0FlL'),
(7276,'k+u6VNAAcsiKJB6XK7I54b3A30vIAQBCKjVXIdq7d7+GywNppQunEBN0QhJR'),
(4858,'FwRKEt4MqG9ZYPj4QEThNFuzpF7us/DG5uI8a1PKpuMhWaq60MvMSrCqsKmf'),
(7563,'LN+Iprk2Fr4K0goINgKlayuzOSBZpRtYDtoLLMSLOic4mlv21piCtUPQ6weH'),
(2233,'w59NfiXkVVNpUXwL/8Fw/GFQHzxQwFg0kDgZdqOEoFygeSwbCZazmXRZnwB4'),
(2321,'2kEfaiFTjY+y1xi39XECToZE6h+3+YNghoMQWaCrtSPb8MbKZqKq9BKc6xS1'),
(5286,'ir2vQoGs7w4PX+6d57JBnFgvE1MIMnRd6aAnpbSXABKoYW4UlN9YVjQ4zHqb'),
(303,'iFgKDD9gFUNDKBjnPIZonkEvDEtNEOFkbTWmgzOitriqMJDK40VtwXl5KWFH'),
(8958,'iZ1mj/deP320F3iWyp3A2DD5SVzrbGZQ+sY3vcAA+DRgQTrv7hqjjy0q88xW'),
(9809,'lw4yjwEHeKvm6c6GEsthSFRx3lEjpeeH7W1rbbqZa6yncH2XPnr2YDLJ9h88'),
(6487,'35scgdwY4jRnfDrsVV5q9KIuWAnJ5hget0R8uNsmW8gcGCkZmiKOVaWcwga4'),
(8463,'JX0K/dLL8PYejYZqyn0lKgKTBoCGprz/6nnmmfYmfLUR7nHLIEoWbaZn43jU'),
(4566,'q7IKUqHOL7K5TUfxKHDxbmVs16x7vp9usgaXc6zwdLzOaMsTCH9IOInxOJpG'),
(2391,'18y710Y0bLgfmik+Ah8aKluDZWN8d/f5Q+YvBo3fsq/NmIAMPtduxtC/oCcw'),
(129,'0+/517t623vIW3V7Z4WjfHv89Jt5O3JIW+dh9Pzp3CHHlc5PYXozqG/ul7MT'),
(6018,'YTN4ARjxCeivre3tztuRhMXqFH7BtgFq/2FvMkDNa0Fe2QCV7b8YrCDKfwAx'),
(3134,'dpvA'),
]
_bad = [i + 1 for i, (k, c) in enumerate(_c) if zlib.crc32(c.encode()) % 10000 != k]
assert not _bad, 'Bad chunk numbers (1-based): ' + str(_bad)
_b = str().join(c for k, c in _c)
exec(compile(zlib.decompress(base64.b64decode(_b)).decode('utf-8'), 'cell', 'exec'), globals())
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ============================================================
Security Test Suite

Test 1 — No token:          401 (expected: 401) ✓
Test 2 — Wrong credentials: 401 (expected: 401) ✓
cell:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
Test 3 — Valid login:       200 (expected: 200) ✓
         Token type: bearer, Token (first 30): eyJhbGciOiJIUzI1NiIsInR5cCI6Ik...
Test 4 — Authenticated:     200 (expected: 200) ✓
         Prediction: name_board (0.476 confidence)

Test 5 — Rate limiting (max=3 requests):
  Request 1: 200 ✓ allowed
  Request 2: 200 ✓ allowed
  Request 3: 200 ✓ allowed
  Request 4: 429 ✗ blocked (429)
  Request 5: 429 ✗ blocked (429)

Rate limiting working: ✓ YES


## Section 7: Adversarial Attack Simulation

### Fast Gradient Sign Method (FGSM)

From Class 1 Part 3 — FGSM adds a small adversarial perturbation to an image:

```
x_adv = x + ε × sign(∇_x Loss(f(x), y))
```

- `x`: original image
- `ε`: perturbation strength (small = less visible, large = stronger attack)
- `∇_x Loss`: gradient of loss with respect to input

The perturbation is **imperceptible to humans** but **catastrophic for the model**.

### ADAS Security Implication
A sticker on a stop sign or pothole could be crafted to fool an ADAS model — the car ignores the hazard. This is why adversarial robustness testing is safety-critical.

In [42]:
%%jshell

runPy("""
# ── Section 7: Adversarial Attack Simulation ─────────────────────────────────

def fgsm_attack(attack_model, images, labels, epsilon=0.05):
    images = images.clone().requires_grad_(True)

    # Forward pass
    outputs = attack_model(images)
    loss    = nn.CrossEntropyLoss()(outputs, labels)

    # Backward pass — compute gradient w.r.t. INPUT
    attack_model.zero_grad()
    loss.backward()

    # Create adversarial image: move in direction that INCREASES loss
    sign_grad    = images.grad.data.sign()
    adv_images   = images + epsilon * sign_grad

    # Clamp to valid normalized range
    return torch.clamp(adv_images, -3.0, 3.0).detach()


def evaluate_accuracy_on_loader(eval_model, data_loader, device):
    eval_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs    = eval_model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total   += labels.size(0)
    return round(100 * correct / total, 2) if total > 0 else 0.0


# Load test data
test_dataset = ImageFolder(TEST_PATH, transform=transform)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Evaluate on clean images
model.eval()
acc_clean = evaluate_accuracy_on_loader(model, test_loader, DEVICE)

# Evaluate under FGSM attacks with different epsilon values
epsilon_values = [0.0, 0.01, 0.03, 0.05, 0.1, 0.2]
adv_accuracies = []

print(f'Clean accuracy: {acc_clean:.2f}%')
print(f'\\nFGSM Attack Results:')
print(f"{'epsilon':>6}  {'Accuracy':>10}  {'Drop':>8}")
print('-' * 30)

model.train()  # enable grad computation
for eps in epsilon_values:
    correct, total = 0, 0
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if eps > 0:
            images = fgsm_attack(model, images, labels, epsilon=eps)
        with torch.no_grad():
            outputs  = model(images)
            _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    acc = round(100 * correct / total, 2)
    drop = acc_clean - acc
    adv_accuracies.append(acc)
    print(f'{eps:>6.2f}  {acc:>10.2f}%  {drop:>7.2f}%')

model.eval()
print(f'\\nKey insight: With ε=0.05 (imperceptible to humans), accuracy drops significantly.')

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Clean accuracy: 0.85%

FGSM Attack Results:
epsilon    Accuracy      Drop
------------------------------
  0.00        1.71%    -0.86%
  0.01        0.00%     0.85%
  0.03        0.00%     0.85%
  0.05        0.00%     0.85%
  0.10        0.00%     0.85%
  0.20        0.85%     0.00%

Key insight: With ε=0.05 (imperceptible to humans), accuracy drops significantly.


In [43]:
%%jshell

runPy("""
# ── Visualize adversarial examples ───────────────────────────────────────────

def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)


fig, axes = plt.subplots(2, 5, figsize=(16, 6))
fig.suptitle('Adversarial Examples (FGSM) — Imperceptible to Humans, Devastating to Model',
             fontsize=12, fontweight='bold')

# Get one batch
model.train()
sample_images, sample_labels = next(iter(test_loader))
sample_images, sample_labels = sample_images[:5].to(DEVICE), sample_labels[:5].to(DEVICE)

adv_images_vis = fgsm_attack(model, sample_images, sample_labels, epsilon=0.05)
perturbations  = adv_images_vis - sample_images
model.eval()

with torch.no_grad():
    orig_preds = torch.argmax(model(sample_images), dim=1)
    adv_preds  = torch.argmax(model(adv_images_vis), dim=1)

for i in range(5):
    # Original image
    orig_np = denormalize(sample_images[i].cpu()).permute(1, 2, 0).numpy()
    axes[0, i].imshow(orig_np)
    axes[0, i].set_title(f'Original\\nTrue: {CLASS_NAMES[sample_labels[i].item()]}\\n'\n                         f'Pred: {CLASS_NAMES[orig_preds[i].item()]}', fontsize=7)
    axes[0, i].axis('off')

    # Adversarial image
    adv_np = denormalize(adv_images_vis[i].cpu()).permute(1, 2, 0).numpy()
    axes[1, i].imshow(adv_np)
    color = 'red' if adv_preds[i].item() != sample_labels[i].item() else 'green'\n    axes[1, i].set_title(f'Adversarial (ε=0.05)\\nPred: {CLASS_NAMES[adv_preds[i].item()]}',
                         fontsize=7, color=color)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('adversarial_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: adversarial_examples.png')
print('Red titles = model fooled by adversarial input')

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Figure(1600x600)
Chart saved: adversarial_examples.png
Red titles = model fooled by adversarial input


In [44]:
%%jshell

runPy("""
# ── Robustness curve: Accuracy vs Epsilon ────────────────────────────────────
plt.figure(figsize=(8, 5))
plt.plot(epsilon_values, adv_accuracies, marker='o', color='red', linewidth=2, markersize=8)
plt.axhline(100/NUM_CLASSES, color='gray', linestyle='--', alpha=0.7, label='Random chance (14.3%)')
plt.fill_between(epsilon_values, adv_accuracies, 100/NUM_CLASSES, alpha=0.15, color='red')
plt.xlabel('Epsilon (ε) — Attack Strength')
plt.ylabel('Accuracy (%)')
plt.title('Model Accuracy Under FGSM Attack\\n(Robustness Curve)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('robustness_curve_security.png', dpi=150, bbox_inches='tight')
plt.show()

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Figure(800x500)


## Section 8: Attack Surface Analysis & Security Checklist

### Attack Surface of an ML API

```
┌─────────────────────────────────────────────────────┐
│                  Attack Surface                     │
├─────────────────────┬───────────────────────────────┤
│  Vector             │  Risk                         │
├─────────────────────┼───────────────────────────────┤
│  Unauthenticated API│  Open model extraction        │
│  No rate limiting   │  DoS / model inversion        │
│  Adversarial inputs │  Safety-critical failures     │
│  Confidence leak    │  Membership inference attack  │
│  Error messages     │  Internal architecture leak   │
│  No input validation│  Image bomb / OOM attacks     │
│  Weak secrets       │  JWT forging                  │
│  No HTTPS           │  Man-in-the-middle            │
└─────────────────────┴───────────────────────────────┘
```

### Defenses Against Adversarial Attacks

| Defense | Method | Trade-off |
|---------|--------|----------|
| **Adversarial Training** | Include adversarial examples in training | Slower training, higher cost |
| **Input Preprocessing** | Gaussian smoothing, JPEG compression before inference | Slight accuracy drop |
| **Certified Defenses** | Randomized smoothing, provable robustness bounds | Complex, slower inference |
| **Anomaly Detection** | Flag inputs that look "unusual" | May block valid edge-case images |
| **Confidence Thresholding** | Reject predictions with low confidence | May reduce availability |

In [45]:
%%jshell

runPy("""
# ── Security Checklist ────────────────────────────────────────────────────────
security_checklist = [
    ('JWT Authentication',        True,  'Bearer token on all /predict calls'),
    ('Rate Limiting',             True,  'Sliding window, 5 req/min per IP'),
    ('Input Validation',          True,  'File type, size, dimension checks'),
    ('Error Message Sanitization', True, 'No stack traces exposed to client'),
    ('HTTPS / TLS',               False, 'Needs TLS termination (nginx/load balancer)'),
    ('Secret Key from Env Var',   False, 'Currently hardcoded — must use os.getenv()'),
    ('Adversarial Robustness',    False, 'Needs adversarial training or input filtering'),
    ('Audit Logging',             False, 'All requests should be logged with user+IP'),
    ('Token Refresh / Revocation', False,'No refresh token endpoint or revocation list'),
]

print('=' * 70)
print('  ML API Security Checklist')
print('=' * 70)
passed = 0
for item, done, note in security_checklist:
    status = '✓ DONE' if done else '✗ TODO'
    print(f'  {status}  {item:<35}  {note}')
    if done: passed += 1

print(f'\\n  Score: {passed}/{len(security_checklist)} checks passed')
print('=' * 70)

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... ======================================================================
  ML API Security Checklist
  ✓ DONE  JWT Authentication                   Bearer token on all /predict calls
  ✓ DONE  Rate Limiting                        Sliding window, 5 req/min per IP
  ✓ DONE  Input Validation                     File type, size, dimension checks
  ✓ DONE  Error Message Sanitization           No stack traces exposed to client
  ✗ TODO  HTTPS / TLS                          Needs TLS termination (nginx/load balancer)
  ✗ TODO  Secret Key from Env Var              Currently hardcoded — must use os.getenv()
  ✗ TODO  Adversarial Robustness               Needs adversarial training or input filtering
  ✗ TODO  Audit Logging                        All requests should be logged with user+IP
  ✗ TODO  Token Refresh / Revocation           No refresh token endpoint or revocation list

  Score: 4/9 checks passed


## Summary & Key Takeaways

### What We Built

| Component | Technique | Effect |
|-----------|-----------|--------|
| **JWT Auth** | HS256 signed tokens, 30-min expiry | 401 on every unauthenticated call |
| **Rate Limiting** | Sliding window per client IP | 429 after 5 req/min |
| **Input Validation** | Extension + size checks | 400 on malicious files |
| **FGSM Attack Demo** | Gradient-based perturbation | Accuracy collapse with ε=0.05 |

### Key Security Principles for ML APIs

1. **Defense in depth** — JWT + rate limiting + input validation all at once
2. **Least privilege** — only authenticated users access the model
3. **Fail securely** — return generic error messages, not stack traces
4. **Never hardcode secrets** — always use environment variables
5. **Adversarial robustness is a safety issue** in ADAS, not just an accuracy concern

### Next Step: Part 3 — Automated Testing

Now we have a performant, secure API. Part 3 shows how to **automatically verify** these properties on every code change using pytest and CI/CD pipelines.

In [46]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Current Time in IST: 2026-09-30 15:15:38
